In [1]:
import subprocess
subprocess.run([
    'pip', 'install', '-q',
    'datasets', 'lxml', 'cairosvg',
    'tokenizers', 'sentencepiece', 'tqdm', 'numpy'
], check=True)
print("Packages installed.")

Packages installed.


In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [3]:
import os, re, json
from lxml import etree
from datasets import load_dataset
from tqdm import tqdm
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.processors import TemplateProcessing

BASE_DIR = '/content/drive/MyDrive/svg-lm-scaling'

DIRS = [
    'data/clean',
    'data/tokenized',
    'tokenizer',
    'checkpoints',
    'logs',
    'samples',
    'configs',
]

for d in DIRS:
    os.makedirs(f'{BASE_DIR}/{d}', exist_ok=True)

print(f"\nAll directories ready under: {BASE_DIR}")


All directories ready under: /content/drive/MyDrive/svg-lm-scaling


In [4]:
VOCAB_SIZE = 4096          # 1K–8K are all reasonable but 4096 is a good balance
CLEAN_DIR = f'{BASE_DIR}/data/clean'
INPUT_FILE = f'{CLEAN_DIR}/all_svgs.jsonl'

# building a plain-text iterator from the jsonl

def svg_iterator(path, limit=None):
    """Yield raw SVG strings one at a time (memory-efficient)."""
    with open(path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if limit and i >= limit:
                break
            try:
                yield json.loads(line)['svg']
            except Exception:
                continue

# Write SVGs to a temp txt file and the tokenizers trainer reads files
TMP_CORPUS = '/tmp/svg_corpus.txt'
print(f"Writing corpus to {TMP_CORPUS} …")
count = 0
with open(TMP_CORPUS, 'w', encoding='utf-8') as out:
    for svg in svg_iterator(INPUT_FILE):
        out.write(svg + '\n')
        count += 1
print(f"Corpus: {count:,} lines written.")

Writing corpus to /tmp/svg_corpus.txt …
Corpus: 774,688 lines written.


In [5]:
# Train BPE
TOK_DIR   = f'{BASE_DIR}/tokenizer'
print(f"\nTraining BPE tokenizer  (vocab_size={VOCAB_SIZE})")

tokenizer = Tokenizer(BPE(unk_token='<unk>'))
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)

trainer = BpeTrainer(
    vocab_size=VOCAB_SIZE,
    special_tokens=['<pad>', '<bos>', '<eos>', '<unk>'],
    min_frequency=2,
    show_progress=True,
)

tokenizer.train(files=[TMP_CORPUS], trainer=trainer)

# Add BOS/EOS wrapping as a post-processor
bos_id = tokenizer.token_to_id('<bos>')
eos_id = tokenizer.token_to_id('<eos>')
tokenizer.post_processor = TemplateProcessing(
    single='<bos> $A <eos>',
    special_tokens=[('<bos>', bos_id), ('<eos>', eos_id)],
)

# Save
tokenizer.save(f'{TOK_DIR}/svg_bpe.json')
print(f"Tokenizer saved to {TOK_DIR}/svg_bpe.json")

# Sanity check
sample = '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24"><path d="M12 2L2 7l10 5 10-5-10-5z"/></svg>'
enc = tokenizer.encode(sample)

print(f"\nVocab size:  {tokenizer.get_vocab_size():,}")
print(f"Sample SVG:  {sample[:80]}…")
print(f"Token count: {len(enc.ids)}")
print(f"Tokens:      {enc.tokens[:20]} …")

# Save token ID constants for use in later steps
config = {
    'vocab_size': tokenizer.get_vocab_size(),
    'pad_id': tokenizer.token_to_id('<pad>'),
    'bos_id': bos_id,
    'eos_id': eos_id,
    'unk_id': tokenizer.token_to_id('<unk>'),
}
with open(f'{TOK_DIR}/token_config.json', 'w') as f:
    json.dump(config, f, indent=2)
print(f"\nToken config: {config}")


Training BPE tokenizer  (vocab_size=4096)
Tokenizer saved to /content/drive/MyDrive/svg-lm-scaling/tokenizer/svg_bpe.json

Vocab size:  4,096
Sample SVG:  <svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24"><path d="M12 2L2 7l1…
Token count: 49
Tokens:      ['<bos>', '<', 'svg', 'Ġxmlns', '="', 'http', '://', 'www', '.', 'w', '3', '.', 'org', '/', '2000', '/', 'svg', '"', 'ĠviewBox', '="'] …

Token config: {'vocab_size': 4096, 'pad_id': 0, 'bos_id': 1, 'eos_id': 2, 'unk_id': 3}


In [6]:
import random
import numpy as np

In [7]:
# Load tokenizer

tokenizer = Tokenizer.from_file(f'{TOK_DIR}/svg_bpe.json')
with open(f'{TOK_DIR}/token_config.json') as f:
    tok_cfg = json.load(f)

PAD_ID = tok_cfg['pad_id']
BOS_ID = tok_cfg['bos_id']
EOS_ID = tok_cfg['eos_id']
print(f"Loaded tokenizer  vocab={tokenizer.get_vocab_size():,}  bos={BOS_ID}  eos={EOS_ID}")

Loaded tokenizer  vocab=4,096  bos=1  eos=2


In [8]:
INPUT_FILE = f'{CLEAN_DIR}/all_svgs.jsonl'
OUT_DIR = f'{BASE_DIR}/data/tokenized'
os.makedirs(OUT_DIR, exist_ok=True)

SEED = 42
BLOCK_SIZE = 512    # context window for training

In [9]:
# Load and tokenize all SVGs

print(f"\nTokenizing SVGs from {INPUT_FILE} …")
all_token_seqs = []   # list of token-id lists (one per SVG)
seq_lengths    = []

with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    lines = f.readlines()

for line in tqdm(lines, desc='Tokenizing'):
    try:
        svg = json.loads(line)['svg']
    except Exception:
        continue

    enc = tokenizer.encode(svg)   # includes <bos>…<eos> via post-processor
    ids = enc.ids

    # Filter sequences that are too long for our context window
    # (keep up to BLOCK_SIZE+1 so we can form one full training example)
    if len(ids) > BLOCK_SIZE + 1:
        continue
    if len(ids) < 4:   # too short to be useful
        continue

    all_token_seqs.append(ids)
    seq_lengths.append(len(ids))

print(f"Valid sequences: {len(all_token_seqs):,}")
print(f"Seq length  min={min(seq_lengths)}  max={max(seq_lengths)}  "
      f"avg={int(sum(seq_lengths)/len(seq_lengths))}")


Tokenizing SVGs from /content/drive/MyDrive/svg-lm-scaling/data/clean/all_svgs.jsonl …


Tokenizing: 100%|██████████| 774688/774688 [17:45<00:00, 727.18it/s]

Valid sequences: 433,737
Seq length  min=86  max=513  avg=342


In [10]:
# 98 / 1 / 1 split (by SVG file, not by token position)

random.seed(SEED)
indices = list(range(len(all_token_seqs)))
random.shuffle(indices)

n = len(indices)
n_val = max(1, int(n * 0.01))
n_test = max(1, int(n * 0.01))
n_train = n - n_val - n_test

train_idx = indices[:n_train]
val_idx = indices[n_train : n_train + n_val]
test_idx = indices[n_train + n_val :]

print(f"\nSplit train={n_train:,}  val={n_val:,}  test={n_test:,} SVGs")

# Packing token sequences and save as numpy uint16 arrays
# Format: flat array of token IDs (same as nanoGPT).
# Concatenating all sequences and the dataloader will slice into BLOCK_SIZE windows.

def pack_and_save(seq_list, filepath):
    """Concatenate token sequences, save as uint16 numpy array."""
    flat = []
    for ids in seq_list:
        flat.extend(ids)
    arr = np.array(flat, dtype=np.uint16)
    np.save(filepath, arr)
    return len(arr)

splits = {
    'train': train_idx,
    'val':   val_idx,
    'test':  test_idx,
}

token_counts = {}
for name, idx_list in splits.items():
    seqs = [all_token_seqs[i] for i in idx_list]
    path = f'{OUT_DIR}/{name}.npy'
    n_tok = pack_and_save(seqs, path)
    token_counts[name] = n_tok
    print(f"{name:5s}: {n_tok:>12,} tokens  in  {path}")


Split train=425,063  val=4,337  test=4,337 SVGs
train:  145,573,946 tokens  in  /content/drive/MyDrive/svg-lm-scaling/data/tokenized/train.npy
val  :    1,482,782 tokens  in  /content/drive/MyDrive/svg-lm-scaling/data/tokenized/val.npy
test :    1,474,488 tokens  in  /content/drive/MyDrive/svg-lm-scaling/data/tokenized/test.npy


In [11]:
# Sequence length histogram

bins = [0, 64, 128, 256, 384, 512, 513]
labels = ['<64', '64-128', '128-256', '256-384', '384-512', '>512(filtered)']
counts = [0] * len(labels)
for l in seq_lengths:
    for i in range(len(bins) - 1):
        if bins[i] <= l < bins[i+1]:
            counts[i] += 1
            break

print("\nSequence length distribution (after filtering):")
for label, cnt in zip(labels, counts):
    print(f"  {label:>16s}: {cnt:>8,}")

# Save stats

stats = {
    'total_svgs': len(all_token_seqs),
    'block_size': BLOCK_SIZE,
    'vocab_size': tokenizer.get_vocab_size(),
    'split': {
        'train_svgs': n_train,
        'val_svgs': n_val,
        'test_svgs': n_test,
    },
    'token_counts':    token_counts,
    'seq_len': {
        'min': int(min(seq_lengths)),
        'max': int(max(seq_lengths)),
        'avg': int(sum(seq_lengths) / len(seq_lengths)),
    },
    'length_histogram': dict(zip(labels, counts)),
}
with open(f'{OUT_DIR}/split_stats.json', 'w') as f:
    json.dump(stats, f, indent=2)

print(f"\nTotal training tokens: {token_counts['train']:,}")
if token_counts['train'] < 100_000_000:
    print("WARNING: training set is below 100M tokens.")
    print("Consider including more data")
else:
    print("Training token target met (>=100M).")


Sequence length distribution (after filtering):
               <64:        0
            64-128:    3,214
           128-256:   95,660
           256-384:  166,175
           384-512:  166,588
    >512(filtered):      975

Total training tokens: 145,573,946
Training token target met (>=100M).


In [12]:
# Decoder-only Transformer model (nanoGPT-style)
# Defines GPTConfig and GPT

import math
import torch
import torch.nn as nn
from torch.nn import functional as F
from dataclasses import dataclass

# Config
@dataclass
class GPTConfig:
    block_size: int = 512
    vocab_size: int = 4096
    n_layer:    int = 4
    n_head:     int = 4
    n_embd:     int = 128
    dropout:    float = 0.0
    bias:       bool = False   # no bias since fewer params, negligible impact to accuracy cost

# Five model sizes
MODEL_CONFIGS = {
    'tiny':   GPTConfig(n_embd=128, n_layer=4,  n_head=4,  dropout=0.0),  # ~1M
    'small':  GPTConfig(n_embd=192, n_layer=6,  n_head=6,  dropout=0.0),  # ~3M
    'medium': GPTConfig(n_embd=384, n_layer=6,  n_head=6,  dropout=0.0),  # ~10M
    'large':  GPTConfig(n_embd=512, n_layer=10, n_head=8,  dropout=0.0),  # ~30M
    'xl':     GPTConfig(n_embd=768, n_layer=12, n_head=12, dropout=0.0),  # ~88M
}

# Ref from https://github.com/karpathy/nanoGPT/blob/master/model.py#L29
# Taken directly from nanoGPT:
# - Overall model structure: GPT, Block, CausalSelfAttention, MLP class layout
# - Weight tying between wte (token embedding) and lm_head
# - The residual projection scaling: std = 0.02 / sqrt(2 * n_layer)
# - _init_weights normal(0, 0.02) for Linear, same for Embedding
# - AdamW with betas=(0.9, 0.95), weight_decay=0.1
# - Gradient clipping at max_norm=1.0
# - The flat .npy data format and get_batch random-offset sampling
# - Flash attention fallback pattern (using F.scaled_dot_product_attention if available, else manual masked attention)

# Modified from nanoGPT:
# - GPTConfig: added MODEL_CONFIGS dict for 5 fixed sizes instead of one (above)
# - num_params() added for the scaling plot
# - generate() added top-p (nucleus) sampling as nanoGPT only has top-k
# - Cosine LR schedule: nanoGPT has it inline in the training loop; we extracted it into a cosine_lr() function reused across steps
# - Training loop: added gradient accumulation, mixed precision (torch.amp), per-step metrics (tokens/s, GPU memory), and safe checkpointing
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn  = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj  = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attn_drop  = nn.Dropout(config.dropout)
        self.resid_drop = nn.Dropout(config.dropout)
        self.n_head  = config.n_head
        self.n_embd  = config.n_embd
        self.dropout = config.dropout
        # Use PyTorch flash attention
        self.flash = hasattr(F, 'scaled_dot_product_attention')
        if not self.flash:
            self.register_buffer(
                'causal_mask',
                torch.tril(torch.ones(config.block_size, config.block_size))
                     .view(1, 1, config.block_size, config.block_size)
            )

    def forward(self, x):
        B, T, C = x.size()
        hs = C // self.n_head
        q, k, v = self.c_attn(x).split(C, dim=2)
        q = q.view(B, T, self.n_head, hs).transpose(1, 2)
        k = k.view(B, T, self.n_head, hs).transpose(1, 2)
        v = v.view(B, T, self.n_head, hs).transpose(1, 2)
        if self.flash:
            y = F.scaled_dot_product_attention(
                q, k, v,
                attn_mask=None,
                dropout_p=self.dropout if self.training else 0.0,
                is_causal=True,
            )
        else:
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(hs))
            att = att.masked_fill(self.causal_mask[:, :, :T, :T] == 0, float('-inf'))
            att = F.softmax(att, dim=-1)
            att = self.attn_drop(att)
            y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.c_proj(y))


class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc   = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu   = nn.GELU()
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.drop   = nn.Dropout(config.dropout)

    def forward(self, x):
        return self.drop(self.c_proj(self.gelu(self.c_fc(x))))


class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd, bias=config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd, bias=config.bias)
        self.mlp  = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x


class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte  = nn.Embedding(config.vocab_size, config.n_embd),
            wpe  = nn.Embedding(config.block_size, config.n_embd),
            drop = nn.Dropout(config.dropout),
            h    = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd, bias=config.bias),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        # Weight tying: embedding and output projection share the same matrix
        self.transformer.wte.weight = self.lm_head.weight
        self.apply(self._init_weights)
        # Scale residual projections by 1/sqrt(2*n_layer) as in GPT-2
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.size()
        assert T <= self.config.block_size, f"Sequence length {T} > block_size {self.config.block_size}"
        pos = torch.arange(T, device=idx.device)
        x = self.transformer.drop(self.transformer.wte(idx) + self.transformer.wpe(pos))
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        else:
            logits = self.lm_head(x[:, [-1], :])
            loss = None
        return logits, loss

    def num_params(self):
        n = sum(p.numel() for p in self.parameters())
        # Subtract embedding weights counted twice due to weight tying
        n -= self.transformer.wte.weight.numel()
        return n

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None, top_p=None, eos_id=None):
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
            if top_p is not None:
                sorted_logits, sorted_idx = torch.sort(logits, descending=True)
                cum_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
                # Remove tokens where cumulative prob exceeds top_p
                remove = cum_probs - F.softmax(sorted_logits, dim=-1) > top_p
                sorted_logits[remove] = float('-inf')
                logits.scatter_(1, sorted_idx, sorted_logits)
            probs = F.softmax(logits, dim=-1)
            next_tok = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_tok), dim=1)
            if eos_id is not None and next_tok.item() == eos_id:
                break
        return idx

# Sanity check

print("\nModel parameter counts:")
print(f"{'Name':>8}  {'Params':>12}  d_model  n_layers  n_heads")
print("-" * 50)
for name, cfg in MODEL_CONFIGS.items():
    m = GPT(cfg)
    print(f"{name:>8}  {m.num_params():>12,}  {cfg.n_embd:>7}  {cfg.n_layer:>8}  {cfg.n_head:>7}")
    del m


Model parameter counts:
    Name        Params  d_model  n_layers  n_heads
--------------------------------------------------
    tiny       853,120      128         4        4
   small     2,755,008      192         6        6
  medium    10,818,432      384         6        6
   large    31,730,176      512        10        8
      xl    85,347,072      768        12       12


In [13]:
# Learning-rate sweep on the Tiny model
# Saves: /results/lr_sweep_tiny.json + /results/best_lr.json

#import sys
#sys.path.insert(0, '/content/drive/MyDrive/svg-lm-scaling')
#from model import *

import os, json, math, time
import numpy as np
import torch
import torch.nn as nn

BASE_DIR = '/content/drive/MyDrive/svg-lm-scaling'
DATA_DIR = f'{BASE_DIR}/data/tokenized'
TOK_DIR  = f'{BASE_DIR}/tokenizer'
RESULTS_DIR = f'{BASE_DIR}/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

# Hyperparameters
BLOCK_SIZE  = 512
MICRO_BATCH = 16     # sequences per micro-step
GRAD_ACCUM  = 8      # effective batch = MICRO_BATCH * GRAD_ACCUM * BLOCK_SIZE tokens
SWEEP_STEPS = 400    # steps per LR candidate (~26M tokens, enough to rank LRs)
WARMUP_FRAC = 0.10   # 10% of sweep steps as linear warmup

LR_CANDIDATES = [1e-4, 3e-4, 6e-4, 1e-3, 3e-3, 6e-3, 1e-2]

# Device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
use_bf16 = device == 'cuda' and torch.cuda.is_bf16_supported()
amp_dtype = torch.bfloat16 if use_bf16 else torch.float16
print(f"Device: {device}  |  AMP dtype: {amp_dtype}")

# Load token config + data
with open(f'{TOK_DIR}/token_config.json') as f:
    tok_cfg = json.load(f)

VOCAB_SIZE = tok_cfg['vocab_size']
BOS_ID = tok_cfg['bos_id']
EOS_ID = tok_cfg['eos_id']

print(f"Vocab size: {VOCAB_SIZE:,}  |  BOS={BOS_ID}  EOS={EOS_ID}")

print("Loading tokenized data:")
train_data = np.load(f'{DATA_DIR}/train.npy', mmap_mode='r')
val_data = np.load(f'{DATA_DIR}/val.npy',   mmap_mode='r')
print(f"train: {len(train_data):,} tokens")
print(f"val: {len(val_data):,} tokens")

# Training utilities

def get_batch(data, batch_size, block_size, device):
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy(data[i:i+block_size].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i+1:i+1+block_size].astype(np.int64)) for i in ix])
    return x.to(device), y.to(device)


def cosine_lr(step, lr_max, lr_min, warmup_steps, total_steps):
    if step < warmup_steps:
        return lr_max * (step + 1) / warmup_steps
    if step >= total_steps:
        return lr_min
    progress = (step - warmup_steps) / (total_steps - warmup_steps)
    return lr_min + 0.5 * (lr_max - lr_min) * (1.0 + math.cos(math.pi * progress))


@torch.no_grad()
def eval_val_loss(model, val_data, num_batches=40):
    model.eval()
    losses = []
    for _ in range(num_batches):
        x, y = get_batch(val_data, MICRO_BATCH, BLOCK_SIZE, device)
        with torch.amp.autocast(device_type=device, dtype=amp_dtype, enabled=(device=='cuda')):
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))


def run_training(config, lr_max, total_steps, seed=42, verbose=True):
    """
    Train a GPT model for total_steps, return val loss curve + final val loss.
    """
    torch.manual_seed(seed)
    model = GPT(config).to(device)

    # Adding compile for speed
    if hasattr(torch, 'compile') and device == 'cuda':
        try:
            model = torch.compile(model)
            print("Compiled model.")
        except Exception:
            pass

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr_max,
        weight_decay=0.1,
        betas=(0.9, 0.95),
        eps=1e-8,
    )
    scaler = torch.amp.GradScaler('cuda', enabled=(device == 'cuda' and not use_bf16))

    warmup_steps = max(1, int(total_steps * WARMUP_FRAC))
    lr_min = lr_max * 0.1

    loss_log = []   # (step, train_loss, val_loss)
    t0 = time.time()
    running_loss = 0.0

    for step in range(total_steps):
        lr_now = cosine_lr(step, lr_max, lr_min, warmup_steps, total_steps)
        for g in optimizer.param_groups:
            g['lr'] = lr_now

        # Gradient accumulation over micro-batches
        optimizer.zero_grad(set_to_none=True)
        step_loss = 0.0
        for _ in range(GRAD_ACCUM):
            x, y = get_batch(train_data, MICRO_BATCH, BLOCK_SIZE, device)
            with torch.amp.autocast(device_type=device, dtype=amp_dtype, enabled=(device=='cuda')):
                _, loss = model(x, y)
                loss = loss / GRAD_ACCUM
            scaler.scale(loss).backward()
            step_loss += loss.item()

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        running_loss += step_loss

        if (step + 1) % 50 == 0:
            avg_train = running_loss / 50
            running_loss = 0.0
            val_loss = eval_val_loss(model, val_data)
            elapsed = time.time() - t0
            tps = (step + 1) * MICRO_BATCH * GRAD_ACCUM * BLOCK_SIZE / elapsed
            loss_log.append({'step': step+1, 'train_loss': avg_train, 'val_loss': val_loss})
            if verbose:
                print(f"step {step+1:4d}/{total_steps}"
                      f"train={avg_train:.4f} val={val_loss:.4f}"
                      f"lr={lr_now:.2e} {tps/1000:.1f}K tok/s")

    final_val = eval_val_loss(model, val_data, num_batches=100)
    total_time = time.time() - t0
    return model, final_val, loss_log, total_time


# LR sweep
RESULT_FILE = f'{RESULTS_DIR}/lr_sweep_tiny.json'

# skip already-evaluated LRs, added this because colab was timing out after long runs
if os.path.exists(RESULT_FILE):
    with open(RESULT_FILE) as f:
        sweep_results = json.load(f)
    done_lrs = {r['lr'] for r in sweep_results}
    print(f"Resuming sweep: {len(done_lrs)}/{len(LR_CANDIDATES)} LRs already done.")
else:
    sweep_results = []
    done_lrs = set()

cfg_tiny = MODEL_CONFIGS['tiny']
cfg_tiny.vocab_size = VOCAB_SIZE
cfg_tiny.block_size = BLOCK_SIZE

print(f"\nSweeping {len(LR_CANDIDATES)} LR values on Tiny model "
      f"({GPT(cfg_tiny).num_params():,} params, {SWEEP_STEPS} steps each):")
print(f"Effective batch size: {MICRO_BATCH * GRAD_ACCUM * BLOCK_SIZE:,} tokens/step\n")

for lr in LR_CANDIDATES:
    if lr in done_lrs:
        print(f"lr={lr:.0e}  (skipping as already done)")
        continue
    print(f"\nlr={lr:.0e}")
    _, final_val, loss_log, elapsed = run_training(cfg_tiny, lr, SWEEP_STEPS, verbose=True)
    result = {'lr': lr, 'final_val_loss': final_val, 'elapsed_s': round(elapsed, 1), 'loss_log': loss_log}
    sweep_results.append(result)
    with open(RESULT_FILE, 'w') as f:
        json.dump(sweep_results, f, indent=2)
    print(f"final val loss = {final_val:.4f}  ({elapsed:.0f}s)")

# Pick best LR
best = min(sweep_results, key=lambda r: r['final_val_loss'])
print(f"LR sweep results:")
for r in sorted(sweep_results, key=lambda r: r['lr']):
    marker = ' BEST' if r['lr'] == best['lr'] else ''         # adding a marker in print to show
    print(f"lr={r['lr']:.0e} val_loss={r['final_val_loss']:.4f}{marker}")
print(f"\nBest LR: {best['lr']:.2e}  (val_loss={best['final_val_loss']:.4f})")

with open(f'{RESULTS_DIR}/best_lr.json', 'w') as f:
    json.dump({'lr': best['lr'], 'val_loss': best['final_val_loss']}, f)

Device: cuda  |  AMP dtype: torch.bfloat16
Vocab size: 4,096  |  BOS=1  EOS=2
Loading tokenized data:
train: 145,573,946 tokens
val: 1,482,782 tokens
Resuming sweep: 7/7 LRs already done.

Sweeping 7 LR values on Tiny model (853,120 params, 400 steps each):
Effective batch size: 65,536 tokens/step

lr=1e-04  (skipping as already done)
lr=3e-04  (skipping as already done)
lr=6e-04  (skipping as already done)
lr=1e-03  (skipping as already done)
lr=3e-03  (skipping as already done)
lr=6e-03  (skipping as already done)
lr=1e-02  (skipping as already done)
LR sweep results:
lr=1e-04 val_loss=3.3572
lr=3e-04 val_loss=1.7060
lr=6e-04 val_loss=1.4273
lr=1e-03 val_loss=1.2840
lr=3e-03 val_loss=1.0891
lr=6e-03 val_loss=1.0393 BEST
lr=1e-02 val_loss=1.2119

Best LR: 6.00e-03  (val_loss=1.0393)


In [14]:
# Training all 5 model sizes for 1 epoch
# Saves checkpoints + /results/scaling_results.json
# Resume-safe: already-trained sizes are skipped automatically.

import os, json, math, time
import numpy as np
import torch

BASE_DIR = '/content/drive/MyDrive/svg-lm-scaling'
DATA_DIR = f'{BASE_DIR}/data/tokenized'
TOK_DIR = f'{BASE_DIR}/tokenizer'
CKPT_DIR = f'{BASE_DIR}/checkpoints'
RESULTS_DIR = f'{BASE_DIR}/results'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

# Hyperparameters

BLOCK_SIZE = 512
MICRO_BATCH = 16
GRAD_ACCUM = 8        # effective batch = MICRO_BATCH x GRAD_ACCUM x BLOCK_SIZE tokens
WARMUP_FRAC = 0.05
LOG_INTERVAL = 100    # log every N steps
EVAL_INTERVAL = 200   # eval val loss every N steps
CKPT_INTERVAL = 200   # save incremental checkpoint every N steps

device = 'cuda' if torch.cuda.is_available() else 'cpu'
use_bf16 = device == 'cuda' and torch.cuda.is_bf16_supported()
amp_dtype = torch.bfloat16 if use_bf16 else torch.float16
print(f"Device: {device} | AMP: {amp_dtype}")

# Load config
with open(f'{TOK_DIR}/token_config.json') as f:
    tok_cfg = json.load(f)
VOCAB_SIZE = tok_cfg['vocab_size']

with open(f'{RESULTS_DIR}/best_lr.json') as f:
    best_lr_cfg = json.load(f)
BEST_LR = best_lr_cfg['lr']
print(f"Using best LR: {BEST_LR:.2e}")

# Load data
print("Loading the data")
train_data = np.load(f'{DATA_DIR}/train.npy', mmap_mode='r')
val_data = np.load(f'{DATA_DIR}/val.npy',   mmap_mode='r')
print(f"train: {len(train_data):,} tokens  |  val: {len(val_data):,} tokens")

STEPS_PER_EPOCH = len(train_data) // (MICRO_BATCH * GRAD_ACCUM * BLOCK_SIZE)
print(f"Steps per epoch (effective batch {MICRO_BATCH*GRAD_ACCUM*BLOCK_SIZE:,} tokens): {STEPS_PER_EPOCH:,}")

# Utilities
def get_batch(data, batch_size, block_size, dev):
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy(data[i:i+block_size].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i+1:i+1+block_size].astype(np.int64)) for i in ix])
    return x.to(dev), y.to(dev)


def cosine_lr(step, lr_max, lr_min, warmup_steps, total_steps):
    if step < warmup_steps:
        return lr_max * (step + 1) / warmup_steps
    if step >= total_steps:
        return lr_min
    progress = (step - warmup_steps) / (total_steps - warmup_steps)
    return lr_min + 0.5 * (lr_max - lr_min) * (1.0 + math.cos(math.pi * progress))


@torch.no_grad()
def eval_val_loss(model, num_batches=60):
    model.eval()
    losses = []
    for _ in range(num_batches):
        x, y = get_batch(val_data, MICRO_BATCH, BLOCK_SIZE, device)
        with torch.amp.autocast(device_type=device, dtype=amp_dtype, enabled=(device=='cuda')):
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))


def _latest_incremental_ckpt(name):
    """
    Return path and step of the latest incremental checkpoint, or (None, 0).
    """
    import glob
    pattern = f'{CKPT_DIR}/sp_{name}_step*.pt'
    paths = sorted(glob.glob(pattern))
    if not paths:
        return None, 0
    latest = paths[-1]
    step = int(os.path.basename(latest).split('step')[1].replace('.pt', ''))
    return latest, step


def train_one_model(name, config, lr, total_steps, seed=42):
    """
    Train one model for total_steps with incremental checkpointing and resume.
    """
    torch.manual_seed(seed)
    model = GPT(config).to(device)

    if hasattr(torch, 'compile') and device == 'cuda':
        try:
            model = torch.compile(model)
        except Exception:
            pass

    params = model.num_params()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=0.1,
        betas=(0.9, 0.95),
        eps=1e-8,
    )
    scaler = torch.amp.GradScaler('cuda', enabled=(device == 'cuda' and not use_bf16))

    warmup_steps = max(1, int(total_steps * WARMUP_FRAC))
    lr_min = lr * 0.1

    # Resume from latest incremental checkpoint if one exists
    resume_path, start_step = _latest_incremental_ckpt(name)
    loss_curve = []
    elapsed_offset = 0.0
    peak_mem_gb = 0.0

    if resume_path:
        print(f"\nResuming {name} from step {start_step:,}  ({resume_path})")
        ckpt_data = torch.load(resume_path, map_location=device)
        model.load_state_dict(ckpt_data['model_state'])
        optimizer.load_state_dict(ckpt_data['optimizer_state'])
        scaler.load_state_dict(ckpt_data['scaler_state'])
        loss_curve = ckpt_data.get('loss_curve', [])
        elapsed_offset = ckpt_data.get('elapsed_s', 0.0)
        peak_mem_gb = ckpt_data.get('peak_mem_gb', 0.0)
    else:
        start_step = 0

    print(f"\n{'='*60}")
    print(f"Training: {name} ({params:,} params)  lr={lr:.2e}  "
          f"steps={start_step:,} {total_steps:,}")
    print(f"{'='*60}")

    running_loss = 0.0
    t0 = time.time()

    for step in range(start_step, total_steps):
        lr_now = cosine_lr(step, lr, lr_min, warmup_steps, total_steps)
        for g in optimizer.param_groups:
            g['lr'] = lr_now

        optimizer.zero_grad(set_to_none=True)
        step_loss = 0.0
        for _ in range(GRAD_ACCUM):
            x, y = get_batch(train_data, MICRO_BATCH, BLOCK_SIZE, device)
            with torch.amp.autocast(device_type=device, dtype=amp_dtype, enabled=(device=='cuda')):
                _, loss = model(x, y)
                loss = loss / GRAD_ACCUM
            scaler.scale(loss).backward()
            step_loss += loss.item()

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        running_loss += step_loss

        if device == 'cuda':
            peak_mem_gb = max(peak_mem_gb, torch.cuda.max_memory_allocated() / 1e9)

        if (step + 1) % LOG_INTERVAL == 0:
            avg_train = running_loss / LOG_INTERVAL
            running_loss = 0.0
            elapsed = elapsed_offset + time.time() - t0
            tps = (step + 1 - start_step) * MICRO_BATCH * GRAD_ACCUM * BLOCK_SIZE / (time.time() - t0)
            loss_curve.append({'step': step+1, 'train_loss': avg_train})
            print(f"  [{step+1:>5}/{total_steps}]  train={avg_train:.4f}  "
                  f"lr={lr_now:.2e}  {tps/1e3:.1f}K tok/s")

        if (step + 1) % EVAL_INTERVAL == 0 or (step + 1) == total_steps:
            val_loss = eval_val_loss(model)
            loss_curve[-1]['val_loss'] = val_loss
            print(f"  ↳ val_loss = {val_loss:.4f}")

        # Incremental checkpoint
        if (step + 1) % CKPT_INTERVAL == 0 and (step + 1) < total_steps:
            inc_path = f'{CKPT_DIR}/sp_{name}_step{step+1}.pt'
            torch.save({
                'config': config.__dict__,
                'model_state': model.state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'scaler_state': scaler.state_dict(),
                'step': step + 1,
                'loss_curve': loss_curve,
                'elapsed_s': elapsed_offset + time.time() - t0,
                'peak_mem_gb': peak_mem_gb,
                'params': params,
            }, inc_path)
            print(f"Incremental checkpoint at {inc_path}")

    wall_time = elapsed_offset + time.time() - t0
    final_val = eval_val_loss(model, num_batches=200)
    tokens_per_sec = total_steps * MICRO_BATCH * GRAD_ACCUM * BLOCK_SIZE / wall_time

    # Final checkpoint (no optimizer state needed — leaner file)
    ckpt_path = f'{CKPT_DIR}/sp_{name}.pt'
    torch.save({
        'config': config.__dict__,
        'model_state': model.state_dict(),
        'final_val_loss': final_val,
        'params': params,
    }, ckpt_path)
    print(f"\nSaved final checkpoint at {ckpt_path}")
    print(f"Final val loss: {final_val:.4f}  |  Time: {wall_time/60:.1f} min  "
          f"|  GPU mem: {peak_mem_gb:.1f} GB  |  Throughput: {tokens_per_sec/1e3:.1f}K tok/s")

    return {
        'name': name,
        'params': params,
        'final_val_loss': final_val,
        'wall_time_s': round(wall_time, 1),
        'tokens_per_sec': round(tokens_per_sec),
        'peak_mem_gb': round(peak_mem_gb, 2),
        'loss_curve': loss_curve,
        'config': {
            'n_embd': config.n_embd,
            'n_layer': config.n_layer,
            'n_head': config.n_head,
        },
    }


# Train all model sizes
RESULT_FILE = f'{RESULTS_DIR}/scaling_results.json'

if os.path.exists(RESULT_FILE):
    with open(RESULT_FILE) as f:
        scaling_results = json.load(f)
    done = {r['name'] for r in scaling_results}
    print(f"\nResuming as {len(done)} model(s) already trained: {done}")
else:
    scaling_results = []
    done = set()

for name, base_cfg in MODEL_CONFIGS.items():
    if name in done:
        print(f"Skipping {name} (already in results).")
        continue

    cfg = GPTConfig(
        block_size = BLOCK_SIZE,
        vocab_size = VOCAB_SIZE,
        n_embd = base_cfg.n_embd,
        n_layer = base_cfg.n_layer,
        n_head = base_cfg.n_head,
        dropout = 0.0,
        bias = False,
    )

    result = train_one_model(name, cfg, BEST_LR, STEPS_PER_EPOCH)
    scaling_results.append(result)

    with open(RESULT_FILE, 'w') as f:
        json.dump(scaling_results, f, indent=2)

# Summary table
print("Scaling Study Summary (Standard Parameterization)")
print(f"{'Model':>8}  {'Params':>12}  {'Val Loss':>9}  {'Time':>8}  {'Tok/s':>8}  {'GPU GB':>7}")
print("-" * 70)
for r in sorted(scaling_results, key=lambda x: x['params']):
    print(f"{r['name']:>8}  {r['params']:>12,}  {r['final_val_loss']:>9.4f}  "
          f"{r['wall_time_s']/60:>7.1f}m  {r['tokens_per_sec']/1e3:>7.1f}K  "
          f"{r['peak_mem_gb']:>7.2f}")


Device: cuda | AMP: torch.bfloat16
Using best LR: 6.00e-03
Loading the data
train: 145,573,946 tokens  |  val: 1,482,782 tokens
Steps per epoch (effective batch 65,536 tokens): 2,221

Resuming as 5 model(s) already trained: {'tiny', 'small', 'xl', 'large', 'medium'}
Skipping tiny (already in results).
Skipping small (already in results).
Skipping medium (already in results).
Skipping large (already in results).
Skipping xl (already in results).
Scaling Study Summary (Standard Parameterization)
   Model        Params   Val Loss      Time     Tok/s   GPU GB
----------------------------------------------------------------------
    tiny       853,120     0.7226     24.0m    101.1K     0.91
   small     2,755,008     0.6502     49.7m     48.8K     1.51
  medium    10,818,432     0.6322    102.7m     23.6K     2.08
   large    31,730,176     1.3587    250.8m      9.7K     4.19
      xl    85,347,072     1.8051    677.8m      3.6K     9.61


In [15]:
# Scaling law analysis and power law fit + plots
# Uses scaling_results.json and creates: scaling_plot.png, training_curves.png

import json, os
import numpy as np
import matplotlib
matplotlib.use('Agg')   # headless
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

BASE_DIR = '/content/drive/MyDrive/svg-lm-scaling'
RESULTS_DIR = f'{BASE_DIR}/results'

# Load results

with open(f'{RESULTS_DIR}/scaling_results.json') as f:
    scaling_results = json.load(f)

scaling_results.sort(key=lambda r: r['params'])

names  = [r['name']           for r in scaling_results]
params = np.array([r['params']           for r in scaling_results], dtype=float)
losses = np.array([r['final_val_loss']   for r in scaling_results], dtype=float)

print("Loaded scaling results:")
print(f"{'Model':>8}  {'Params':>12}  {'Val Loss':>10}")
for n, p, l in zip(names, params, losses):
    print(f"{n:>8}  {p:>12,.0f}  {l:>10.4f}")

# Power-law fit
# Strategy:
#   1. Log-log linear regression (most stable, no free c, 2 params)
#   2. 3-parameter curve_fit with c fixed to min(loss)*0.95 as a cross-check
# With only 5 points and 3 free params the covariance of the full fit
# diverges when losses are close together (degenerate a/alpha/c).

def power_law(N, a, alpha, c):
    return a * N**(-alpha) + c

def power_law_2(N, a, alpha):
    return a * N**(-alpha)

# log-log linear regression (L = a·N^{-alpha}, no offset)
log_N = np.log10(params)
log_L = np.log10(losses)
slope, intercept = np.polyfit(log_N, log_L, 1)
alpha_loglog = -slope
a_loglog     = 10 ** intercept

y_pred_ll = power_law_2(params, a_loglog, alpha_loglog)
ss_res = np.sum((losses - y_pred_ll)**2)
ss_tot = np.sum((losses - losses.mean())**2)
r2_loglog = 1 - ss_res / ss_tot

print(f"\nLog-log regression (L = a x N^{{-alpha}}, no offset):")
print(f"L = {a_loglog:.3f} · N^({alpha_loglog:.4f})")
print(f"alpha = {alpha_loglog:.4f}   R2 = {r2_loglog:.4f}")
# Compare: Kaplan NLP ≈ 0.076 and Chinchilla ≈ 0.34

# Method 2: 3-parameter fit with c fixed
# Fix c = 95% of the minimum observed loss (irreducible floor estimate)
# so a and alpha have a well-defined signal to fit against.
c_fixed = losses.min() * 0.95
losses_shifted = losses - c_fixed   # must be positive for log-space fitting

try:
    popt2, pcov2 = curve_fit(
        power_law_2, params, losses_shifted,
        p0=[a_loglog, alpha_loglog],
        bounds=([0, 0], [1e8, 2.0]),
        maxfev=10000,
    )
    a_fit, alpha_fit = popt2
    c_fit = c_fixed
    perr2 = np.sqrt(np.diag(pcov2))
    y_pred = power_law(params, a_fit, alpha_fit, c_fit)
    ss_res = np.sum((losses - y_pred)**2)
    r2 = 1 - ss_res / ss_tot
    fit_ok = True
    print(f"\n3-param fit (c fixed = {c_fixed:.4f}):")
    print(f"L = {a_fit:.3f} · N^(-{alpha_fit:.4f}) + {c_fit:.4f}")
    print(f"a +- {perr2[0]:.3f},  alpha +- {perr2[1]:.4f}  R2 = {r2:.4f}")
except Exception as e:
    # Fall back to log-log result
    a_fit, alpha_fit, c_fit = a_loglog, alpha_loglog, 0.0
    perr2 = [0.0, 0.0]
    y_pred = power_law_2(params, a_fit, alpha_fit)
    r2 = r2_loglog
    fit_ok = True
    print(f"3-param fit failed ({e}) using log-log result.")

# Use log-log alpha as the primary reported value (more robust)
alpha_report = alpha_loglog
r2_report = r2_loglog
print(f"\nReporting alpha = {alpha_report:.4f}  (log-log, most stable with n=5 points)")

fit_results = {
    'a': float(a_fit), 'alpha': float(alpha_fit), 'c': float(c_fit),
    'alpha_loglog': float(alpha_loglog), 'a_loglog': float(a_loglog),
    'r2_loglog': float(r2_loglog), 'r2_3param': float(r2),
    'parameterization': 'SP',
}
with open(f'{RESULTS_DIR}/power_law_fit_sp.json', 'w') as f:
    json.dump(fit_results, f, indent=2)

# Scaling plot

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(params, losses, s=80, zorder=5, label='Measured', color='steelblue')
for n, p, l in zip(names, params, losses):
    ax.annotate(n, (p, l), textcoords='offset points', xytext=(6, 4), fontsize=9)

if fit_ok:
    N_range = np.logspace(np.log10(params.min()*0.5), np.log10(params.max()*3), 300)
    ax.plot(N_range, power_law_2(N_range, a_loglog, alpha_loglog), 'r--', lw=1.5,
            label=fr'Fit: $L = {a_loglog:.2f} \cdot N^{{-{alpha_loglog:.3f}}}$  ($\alpha={alpha_loglog:.3f}$)')

ax.set_xscale('log')
ax.set_xlabel('Number of Parameters (log scale)')
ax.set_ylabel('Validation Loss (1 epoch)')
ax.set_title('SVG LM Scaling Law (Standard Parameterization)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/scaling_plot_sp.png', dpi=150)
plt.show()
print(f"Saved to {RESULTS_DIR}/scaling_plot_sp.png")

# Training curves

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax1, ax2 = axes
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(scaling_results)))

for r, color in zip(scaling_results, colors):
    curve = r.get('loss_curve', [])
    if not curve:
        continue
    steps = [p['step'] for p in curve]
    train_losses = [p['train_loss'] for p in curve]
    val_losses = [p.get('val_loss') for p in curve]

    ax1.plot(steps, train_losses, label=r['name'], color=color)
    vl = [(s, v) for s, v in zip(steps, val_losses) if v is not None]
    if vl:
        ax2.plot([s for s,_ in vl], [v for _,v in vl], label=r['name'], color=color)

ax1.set_xlabel('Step')
ax1.set_ylabel('Training Loss')
ax1.set_title('Training Loss Curves')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.set_xlabel('Step')
ax2.set_ylabel('Validation Loss')
ax2.set_title('Validation Loss Curves')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/training_curves_sp.png', dpi=150)
plt.show()
print(f"Saved to {RESULTS_DIR}/training_curves_sp.png")

# Summary

print("\n" + "="*60)
print("Summary")
print("="*60)
if fit_ok:
    print(f"Scaling exponent alpha= {alpha_report:.4f}  (log-log linear regression)")
    print(f"R2 (log-log) = {r2_report:.4f}")
    print()
    print("Interpretation:")
    factor = 10**(-alpha_report)
    print(f"alpha = {alpha_report:.3f}: a 10x increase in parameters multiplies loss by 10^(-alpha) = {factor:.3f}")
    print(f"~{(1-factor)*100:.1f}% reduction per multiples of 10 of parameters.")
    print()
    if alpha_report < 0.05:
        print("Very low alpha indicating weak scaling implying that model size matters little in this regime.")
    elif alpha_report < 0.15:
        print("alpha near NLP range (Kaplan ≈ 0.076) showing consistent scaling behaviour.")
    elif alpha_report < 0.5:
        print("alpha above NLP range implying that SVG structure may be more learnable and stronger returns.")
    else:
        print("High alpha shows loss varies sharply with N and likely still on the steep part of the curve.")


Loaded scaling results:
   Model        Params    Val Loss
    tiny       853,120      0.7226
   small     2,755,008      0.6502
  medium    10,818,432      0.6322
   large    31,730,176      1.3587
      xl    85,347,072      1.8051

Log-log regression (L = a x N^{-alpha}, no offset):
L = 0.030 · N^(-0.2139)
alpha = -0.2139   R2 = 0.7723
3-param fit failed (Initial guess is outside of provided bounds) using log-log result.

Reporting alpha = -0.2139  (log-log, most stable with n=5 points)
Saved to /content/drive/MyDrive/svg-lm-scaling/results/scaling_plot_sp.png
Saved to /content/drive/MyDrive/svg-lm-scaling/results/training_curves_sp.png

Summary
Scaling exponent alpha= -0.2139  (log-log linear regression)
R2 (log-log) = 0.7723

Interpretation:
alpha = -0.214: a 10x increase in parameters multiplies loss by 10^(-alpha) = 1.636
~-63.6% reduction per multiples of 10 of parameters.

Very low alpha indicating weak scaling implying that model size matters little in this regime.
